In [2]:
import pandas as pd
import numpy as np
import itertools
import networkx as nx
import matplotlib.pyplot as plt


### Class

In [73]:
import itertools

def all_rankings(people_names):
    """
    Return all weak rankings over people_names.
    Each ranking is a dict {person: rank}, rank starts from 1.
    Indifferences are allowed.
    """
    n = len(people_names)
    results = []

    def generate_rgs(n):
        if n == 0:
            yield []
            return
        for rgs in generate_rgs(n - 1):
            max_label = max(rgs) if rgs else -1
            for k in range(max_label + 2):
                yield rgs + [k]

    for rgs in generate_rgs(n):
        blocks = {}
        for person, label in zip(people_names, rgs):
            blocks.setdefault(label, []).append(person)

        for perm in itertools.permutations(blocks.keys()):
            ranking = {}
            for rank, block_label in enumerate(perm, start=1):
                for person in blocks[block_label]:
                    ranking[person] = rank
            results.append(ranking)

    return results


def all_profiles(people_names):
    """
    Return all possible preference profiles.
    Each profile is a dict:
        { person_i : ranking_i }
    where ranking_i is a weak preference over people_names \ {person_i}.
    """
    individual_preferences = {}

    # enumerate all weak preferences for each person over others
    for person in people_names:
        others = [p for p in people_names if p != person]
        individual_preferences[person] = all_rankings(others)

    profiles = []

    # Cartesian product of all individuals' preference sets
    for pref_tuple in itertools.product(
        *[individual_preferences[p] for p in people_names]
    ):
        profile = {
            person: pref
            for person, pref in zip(people_names, pref_tuple)
        }
        profiles.append(profile)

    return profiles

In [77]:
def all_profiles_including_self(people_names):
    """
    Return all possible preference profiles.
    Each profile is a dict:
        { person_i : ranking_i }
    where ranking_i is a weak preference over people_names \ {person_i}.
    """
    preferences = all_rankings(people_names)
    profiles = []
    for pref_tuple in itertools.product(
        *[preferences for _ in people_names]
    ):
        profile = {
            person: pref
            for person, pref in zip(people_names, pref_tuple)
        }
        profiles.append(profile)
    return profiles


In [55]:
def is_blocking(person_name, matching, profile):
    """
    Check if person_name forms a blocking pair in the given matching.
    Return True if there exists another person that he prefers to and who weakly prefers him back.
    """
    matched_with = matching[person_name]
    person_preferences = profile[person_name]
    for candidate in profile.keys():
        if candidate == person_name or candidate == matched_with:
            continue
        # Check if person_name prefers candidate to matched_with
        if person_preferences[candidate] < person_preferences[matched_with]:
            candidate_preferences = profile[candidate]
            candidate_matched_with = matching[candidate]
            # Check if candidate weakly prefers person_name to their match
            if candidate_preferences[person_name] <= candidate_preferences[candidate_matched_with]:
                print(f"{person_name} blocks with {candidate}")
                print(f"rank of matched_with: {person_preferences[matched_with]}, rank of candidate: {person_preferences[candidate]}")
                print(f"rank of matched_with for candidate: {candidate_preferences[candidate_matched_with]}, rank of person_name: {candidate_preferences[person_name]}")
                return True
    return False

In [81]:
def is_p_blocking(person_name, matching, profile):
    """
    Check if person_name forms a blocking pair in the given matching.
    Return True if there exists another person that he prefers to and who weakly prefers him back.
    """
    matched_with = matching[person_name]
    person_preferences = profile[person_name]
    for candidate in profile.keys():
        if candidate == person_name or candidate == matched_with:
            continue
        # Check if person_name prefers candidate to matched_with
        if person_preferences[candidate] < person_preferences[matched_with]:
            candidate_preferences = profile[candidate]
            candidate_matched_with = matching[candidate]
            # Check if candidate weakly prefers person_name to their match
            if candidate_preferences[person_name] < candidate_preferences[candidate_matched_with]:
                #print(f"{person_name} blocks with {candidate}")
                #print(f"rank of matched_with: {person_preferences[matched_with]}, rank of candidate: {person_preferences[candidate]}")
                #print(f"rank of matched_with for candidate: {candidate_preferences[candidate_matched_with]}, rank of person_name: {candidate_preferences[person_name]}")
                return True
    return False

In [78]:
def people_decreaced(matching1, matching2, profile):
    """
    Return the list of people who strictly decreased their match from matching1 to matching2.
    """
    decreased_people = []
    for person in matching1.keys():
        match1 = matching1[person]
        match2 = matching2[person]
        if profile[person][match2] > profile[person][match1]:
            decreased_people.append(person)
    return decreased_people

In [35]:
def is_pn_dominating(matching1, matching2, profile):
    """
    Check if matching2 priority-neutral dominates matching1.
    """
    decreased_people = people_decreaced(matching1, matching2, profile)
    for person in decreased_people:
        if is_blocking(person, matching2, profile):
            return False
    return True

In [ ]:
def is_p_pn_dominating(matching1, matching2, profile):
    """
    Check if matching2 priority-neutral dominates matching1.
    """
    decreased_people = people_decreaced(matching1, matching2, profile)
    for person in decreased_people:
        if is_p_blocking(person, matching2, profile):
            return False
    return True

In [84]:
people_names = ['i1', 'i2', 'i3', 'i4']
m_1 = {'i1': 'i2', 'i2': 'i1', 'i3': 'i4', 'i4': 'i3'}
m_2 = {'i1': 'i3', 'i3': 'i1', 'i2': 'i4', 'i4': 'i2'}
m_3 = {'i1': 'i4', 'i4': 'i1', 'i2': 'i3', 'i3': 'i2'}

In [85]:
all_profiles_four = all_profiles(people_names)

In [44]:
from tqdm import tqdm

for profile in tqdm(all_profiles_four, desc="Scanning profiles"):
    if (
        is_pn_dominating(m_2, m_1, profile)
        and is_pn_dominating(m_3, m_2, profile)
        and is_pn_dominating(m_1, m_3, profile)
    ):
        print("Found a cycle in PN dominance for profile:")
        print(profile)
        blockings_m1 = [p for p in people_names if is_blocking(p, m_1, profile)]
        blockings_m2 = [p for p in people_names if is_blocking(p, m_2, profile)]
        blockings_m3 = [p for p in people_names if is_blocking(p, m_3, profile)]
        print("Blocking people in m_1:", blockings_m1)
        print("Blocking people in m_2:", blockings_m2)
        print("Blocking people in m_3:", blockings_m3)

Scanning profiles: 100%|██████████| 28561/28561 [00:00<00:00, 588972.72it/s]

Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i1': 1, 'i2': 1, 'i3': 1}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i4': 1, 'i1': 2, 'i3': 2}, 'i3': {'i2': 1, 'i1': 2, 'i4': 2}, 'i4': {'i3': 1, 'i1': 2, 'i2': 2}}
Blocking people in m_1: ['i3']
Blocking people in m_2: ['i4']
Blocking people in m_3: ['i2']
Found a cycle in PN dominance for profile:
{'i1': {'i4': 1, 'i2': 2, 'i3': 2}, 'i2': {'i1': 1, 'i3': 2, 'i4': 2}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i2': 1, 'i1': 2, 'i3': 2}}
Blocking people in m_1: ['i1']
Blocking people in m_2: ['i2']
Blocking people in m_3: ['i4']
Found a cycle in PN dominance for profile:
{'i1': {'i3': 1, 'i2': 2, 'i4': 2}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i4': 1, 'i1': 2, 'i2': 2}, 'i4': {'i1': 1, 'i2': 2, 'i3': 2

In [86]:
for profile in tqdm(all_profiles_four, desc="Scanning profiles"):
    if (
        is_p_pn_dominating(m_2, m_1, profile)
        and is_p_pn_dominating(m_3, m_2, profile)
        and is_p_pn_dominating(m_1, m_3, profile)
    ):
        print("Found a cycle in PN dominance for profile:")
        print(profile)
        blockings_m1 = [p for p in people_names if is_p_blocking(p, m_1, profile)]
        blockings_m2 = [p for p in people_names if is_p_blocking(p, m_2, profile)]
        blockings_m3 = [p for p in people_names if is_p_blocking(p, m_3, profile)]
        print("Blocking people in m_1:", blockings_m1)
        print("Blocking people in m_2:", blockings_m2)
        print("Blocking people in m_3:", blockings_m3)

Scanning profiles: 100%|██████████| 28561/28561 [00:00<00:00, 235626.62it/s]

Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i1': 1, 'i2': 1, 'i3': 1}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i1': 1, 'i2': 1, 'i3': 2}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i3': 1, 'i1': 2, 'i2': 2}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i2': 1, 'i3': 1, 'i4': 1}, 'i2': {'i1': 1, 'i3': 1, 'i4': 1}, 'i3': {'i1': 1, 'i2': 1, 'i4': 1}, 'i4': {'i1': 1, 'i3': 1, 'i2': 2}}
Blocking people in m_

In [83]:
people_names = ['i1', 'i2', 'i3']
m_1 = {'i1': 'i2', 'i2': 'i1', 'i3': 'i3'}
m_2 = {'i1': 'i3', 'i3': 'i1', 'i2': 'i2'}
m_3 = {'i1': 'i1', 'i2': 'i3', 'i3': 'i2'}
all_profiles_three = all_profiles_including_self(people_names)
for profile in tqdm(all_profiles_three, desc="Scanning profiles"):
    if (
        is_p_pn_dominating(m_2, m_1, profile)
        and is_p_pn_dominating(m_3, m_2, profile)
        and is_p_pn_dominating(m_1, m_3, profile)
    ):
        print("Found a cycle in PN dominance for profile:")
        print(profile)
        blockings_m1 = [p for p in people_names if is_p_blocking(p, m_1, profile)]
        blockings_m2 = [p for p in people_names if is_p_blocking(p, m_2, profile)]
        blockings_m3 = [p for p in people_names if is_p_blocking(p, m_3, profile)]
        print("Blocking people in m_1:", blockings_m1)
        print("Blocking people in m_2:", blockings_m2)
        print("Blocking people in m_3:", blockings_m3)

Scanning profiles: 100%|██████████| 2197/2197 [00:00<00:00, 89739.36it/s]

Found a cycle in PN dominance for profile:
{'i1': {'i1': 1, 'i2': 1, 'i3': 1}, 'i2': {'i1': 1, 'i2': 1, 'i3': 1}, 'i3': {'i1': 1, 'i2': 1, 'i3': 1}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i1': 1, 'i2': 1, 'i3': 1}, 'i2': {'i1': 1, 'i2': 1, 'i3': 1}, 'i3': {'i1': 1, 'i2': 1, 'i3': 2}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i1': 1, 'i2': 1, 'i3': 1}, 'i2': {'i1': 1, 'i2': 1, 'i3': 1}, 'i3': {'i3': 1, 'i1': 2, 'i2': 2}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i1': 1, 'i2': 1, 'i3': 1}, 'i2': {'i1': 1, 'i2': 1, 'i3': 1}, 'i3': {'i1': 1, 'i3': 1, 'i2': 2}}
Blocking people in m_1: []
Blocking people in m_2: []
Blocking people in m_3: []
Found a cycle in PN dominance for profile:
{'i1': {'i1': 1, 'i2': 1, 'i3': 1}, '